# 📧 Notebook 01: Preprocessing Pipeline & Manual Calculation
## Proyek Text Mining: Deteksi Spam SMS/Email

---

**Kelompok:** [Nama Kelompok]  
**Anggota:** [Nama Anggota 1] | [Nama Anggota 2] | [Nama Anggota 3]  
**Tanggal:** Oktober 2024  

---

### 🎯 Tujuan Notebook
1. **Eksplorasi Data (EDA)** — Memahami distribusi dan karakteristik dataset
2. **Preprocessing Pipeline** — Tokenisasi, normalisasi, stopword removal, dan penanganan dirty text
3. **Perhitungan Manual** — BoW, TF-IDF, dan Cosine Similarity pada 3 dokumen sampel
4. **Setup Eksperimen** — Persiapan 2 eksperimen untuk Milestone 2

---

## 📦 1. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import math
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Styling matplotlib
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor'] = '#16213e'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ Library berhasil diimport!')
print(f'   Pandas  : {pd.__version__}')
print(f'   NumPy   : {np.__version__}')

In [ ]:
# Load dataset
df = pd.read_csv('../data/raw_data.csv')
df_test = pd.read_csv('../data/test_cases.csv')

print('='*60)
print('📊 INFO DATASET UTAMA')
print('='*60)
print(f'  Total baris  : {len(df)}')
print(f'  Kolom        : {list(df.columns)}')
print(f'  Distribusi   : {df["label"].value_counts().to_dict()}')
print()
print('='*60)
print('🧪 INFO DATA UJI')
print('='*60)
print(f'  Total baris  : {len(df_test)}')
print()

df.head()

## 📊 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('📊 Eksplorasi Dataset — Spam vs Ham', fontsize=16, fontweight='bold', color='#00d4aa', y=1.02)

# Plot 1: Distribusi label
label_counts = df['label'].value_counts()
colors = ['#ff6b6b', '#51cf66']
wedges, texts, autotexts = axes[0].pie(
    label_counts.values,
    labels=label_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(12)
    autotext.set_fontweight('bold')
axes[0].set_title('Distribusi Label', fontsize=13, fontweight='bold', pad=15)

# Plot 2: Panjang karakter pesan
df['panjang_karakter'] = df['pesan'].str.len()
for label, color in zip(['spam', 'ham'], ['#ff6b6b', '#51cf66']):
    data_label = df[df['label'] == label]['panjang_karakter']
    axes[1].hist(data_label, alpha=0.7, bins=15, color=color, label=label, edgecolor='white')
axes[1].set_xlabel('Panjang Karakter')
axes[1].set_ylabel('Frekuensi')
axes[1].set_title('Distribusi Panjang Pesan', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Plot 3: Jumlah kata
df['jumlah_kata'] = df['pesan'].str.split().str.len()
spam_data = df[df['label']=='spam']['jumlah_kata']
ham_data = df[df['label']=='ham']['jumlah_kata']
bp = axes[2].boxplot([spam_data, ham_data],
    labels=['SPAM', 'HAM'],
    patch_artist=True,
    boxprops=dict(facecolor='#ff6b6b', alpha=0.7),
    medianprops=dict(color='white', linewidth=2)
)
bp['boxes'][1].set_facecolor('#51cf66')
axes[2].set_ylabel('Jumlah Kata')
axes[2].set_title('Distribusi Jumlah Kata', fontsize=13, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/eda_visualization.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e', edgecolor='none')
plt.show()

print(f'\n📈 Statistik Deskriptif:')
print(df.groupby('label')[['panjang_karakter', 'jumlah_kata']].describe().round(2))

## 🛠️ 3. Preprocessing Pipeline

### Alur Preprocessing:
```
Raw Text
   ↓
[1] Case Folding (lowercase)
   ↓
[2] Cleaning (hapus URL, mention, angka, simbol)
   ↓
[3] Normalisasi (slang → baku, typo → benar)
   ↓
[4] Tokenisasi (split menjadi token kata)
   ↓
[5] Stopword Removal
   ↓
[6] Stemming/Lemmatisasi
   ↓
Clean Tokens → Feature Extraction
```

In [ ]:
# ============================================================
# KAMUS NORMALISASI SLANG & TYPO
# ============================================================
kamus_normalisasi = {
    # Singkatan umum
    'gak': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak',
    'gk': 'tidak', 'g': 'tidak', 'tdk': 'tidak', 'krn': 'karena',
    'blm': 'belum', 'udh': 'sudah', 'sdh': 'sudah', 'yg': 'yang',
    'utk': 'untuk', 'dgn': 'dengan', 'bgt': 'banget', 'bngt': 'banget',
    'hrs': 'harus', 'bisa': 'bisa', 'sy': 'saya', 'ak': 'aku',
    'aku': 'aku', 'kita': 'kita', 'km': 'kamu', 'kmu': 'kamu',
    'lg': 'lagi', 'lgi': 'lagi', 'sm': 'sama', 'trs': 'terus',
    'jg': 'juga', 'juga': 'juga', 'tp': 'tapi', 'tpi': 'tapi',
    'dr': 'dari', 'pd': 'pada', 'bs': 'bisa', 'br': 'baru',
    'nih': 'ini', 'tuh': 'itu', 'banget': 'banget', 'emg': 'memang',
    'emang': 'memang', 'dpt': 'dapat', 'dtg': 'datang', 'bk': 'buka',
    # Typo spam umum
    'vrifikasi': 'verifikasi', 'rkening': 'rekening', 'mnlak': 'melalui',
    'brkut': 'berikut', 'sgr': 'segera', 'penting': 'penting',
    'grts': 'gratis', 'grtis': 'gratis', 'rbu': 'ribu', 'jt': 'juta',
    # Bahasa gaul digital
    'saldo': 'saldo', 'transfer': 'transfer', 'rek': 'rekening',
    'no': 'nomor', 'hp': 'handphone', 'wa': 'whatsapp', 'tlp': 'telepon',
    'hbd': 'selamat ulang tahun', 'asap': 'segera',
}

# ============================================================
# STOPWORD BAHASA INDONESIA (custom)
# ============================================================
stopwords_id = set([
    'yang', 'dan', 'di', 'ke', 'dari', 'ini', 'itu', 'dengan', 'untuk',
    'pada', 'adalah', 'juga', 'dalam', 'tidak', 'ada', 'akan', 'oleh',
    'karena', 'sudah', 'saya', 'kamu', 'kami', 'mereka', 'anda', 'dia',
    'atau', 'tapi', 'tetapi', 'namun', 'jika', 'kalau', 'maka', 'agar',
    'supaya', 'bahwa', 'sehingga', 'yaitu', 'yakni', 'seperti', 'antara',
    'bisa', 'bagi', 'sebagai', 'lagi', 'pun', 'saja', 'masih', 'paling',
    'sangat', 'lebih', 'belum', 'juga', 'saat', 'ketika', 'selama',
    'sering', 'hanya', 'jangan', 'mau', 'maka', 'aku', 'kita',
    'sebuah', 'setiap', 'semua', 'telah', 'dengan', 'secara',
    # Stopwords English dasar
    'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall', 'can', 'to', 'of',
    'in', 'for', 'on', 'with', 'at', 'by', 'from', 'as', 'or', 'and',
    'but', 'if', 'not', 'that', 'this', 'it', 'its', 'you', 'your',
    'we', 'our', 'they', 'their', 'i', 'my', 'me', 'he', 'she',
    'his', 'her', 'him', 'us', 'them', 'so', 'just', 'up',
])

print('✅ Kamus normalisasi dimuat:', len(kamus_normalisasi), 'entri')
print('✅ Daftar stopword dimuat:', len(stopwords_id), 'kata')

In [ ]:
# ============================================================
# FUNGSI PREPROCESSING PIPELINE
# ============================================================

def step1_case_folding(teks):
    """Langkah 1: Ubah semua teks ke lowercase"""
    return teks.lower()

def step2_cleaning(teks):
    """Langkah 2: Hapus noise (URL, email, mention, angka, tanda baca)"""
    # Hapus URL
    teks = re.sub(r'http[s]?://\S+|www\.\S+|bit\.ly/\S+', ' URL ', teks)
    # Hapus email
    teks = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', ' EMAIL ', teks)
    # Hapus nomor telepon (format Indonesia)
    teks = re.sub(r'(?:\+62|0)[2-9][0-9]{6,11}', ' NOMOR_TELPON ', teks)
    # Hapus nomor rekening (angka 10+ digit)
    teks = re.sub(r'\b\d{10,}\b', ' NOMOR_REKENING ', teks)
    # Hapus tanda seru berulang dan simbol khusus spam
    teks = re.sub(r'!+', '!', teks)
    teks = re.sub(r'\?+', '?', teks)
    # Hapus karakter non-alfanumerik (selain spasi)
    teks = re.sub(r'[^a-z0-9\s]', ' ', teks)
    # Hapus angka tunggal
    teks = re.sub(r'\b\d+\b', '', teks)
    # Normalisasi spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

def step3_normalisasi(teks):
    """Langkah 3: Normalisasi slang dan typo menggunakan kamus"""
    kata_kata = teks.split()
    kata_normalisasi = [kamus_normalisasi.get(kata, kata) for kata in kata_kata]
    return ' '.join(kata_normalisasi)

def step4_tokenisasi(teks):
    """Langkah 4: Tokenisasi (split menjadi daftar token)"""
    return teks.split()

def step5_stopword_removal(tokens):
    """Langkah 5: Hapus stopword"""
    return [token for token in tokens if token not in stopwords_id and len(token) > 1]

def step6_stemming_sederhana(tokens):
    """Langkah 6: Stemming sederhana (hapus prefiks/sufiks umum bahasa Indonesia)"""
    prefixes = ['me', 'ber', 'ter', 'pe', 'ke', 'se', 'di', 'ke']
    suffixes = ['kan', 'an', 'nya', 'i']
    hasil = []
    for token in tokens:
        # Hapus prefiks
        for pref in prefixes:
            if token.startswith(pref) and len(token) > len(pref) + 2:
                token = token[len(pref):]
                break
        # Hapus sufiks
        for suf in suffixes:
            if token.endswith(suf) and len(token) > len(suf) + 2:
                token = token[:-len(suf)]
                break
        hasil.append(token)
    return hasil

def full_preprocessing_pipeline(teks, verbose=False):
    """Pipeline lengkap dari raw text ke clean tokens"""
    original = teks
    
    s1 = step1_case_folding(teks)
    s2 = step2_cleaning(s1)
    s3 = step3_normalisasi(s2)
    s4 = step4_tokenisasi(s3)
    s5 = step5_stopword_removal(s4)
    s6 = step6_stemming_sederhana(s5)
    
    if verbose:
        print(f'  📥 Input      : {original[:80]}...' if len(original) > 80 else f'  📥 Input      : {original}')
        print(f'  [1] Lowercase : {s1[:80]}' if len(s1) > 80 else f'  [1] Lowercase : {s1}')
        print(f'  [2] Cleaning  : {s2[:80]}' if len(s2) > 80 else f'  [2] Cleaning  : {s2}')
        print(f'  [3] Normalis  : {s3[:80]}' if len(s3) > 80 else f'  [3] Normalis  : {s3}')
        print(f'  [4] Tokens    : {s4[:10]}' + ('...' if len(s4) > 10 else ''))
        print(f'  [5] No-Stop   : {s5[:10]}' + ('...' if len(s5) > 10 else ''))
        print(f'  [6] Stemmed   : {s6[:10]}' + ('...' if len(s6) > 10 else ''))
    
    return s6

print('✅ Semua fungsi preprocessing didefinisikan!')

In [ ]:
# ============================================================
# DEMONSTRASI PENANGANAN DIRTY TEXT
# ============================================================
dirty_samples = [
    {
        'nama': '🔴 DIRTY TEXT 1: Typo & Singkatan Spam',
        'teks': 'Ythg bpk/ibu, vrifikasi rkening Anda mnlak kna tautan brkut: http://bit.ly/palsu'
    },
    {
        'nama': '🟡 DIRTY TEXT 2: Nomor Telepon & Rekening',
        'teks': 'Transfer Rp 500.000 ke rek 1234567890123 a.n. John. Hubungi 081234567890 segera!'
    },
    {
        'nama': '🟢 DIRTY TEXT 3: Capslock & Karakter Berulang',
        'teks': 'SELAMAT!!! ANDA MENANG!!! KLIK SEKARANG!!! PROMO TERBATAS!!!'
    },
    {
        'nama': '🔵 DIRTY TEXT 4: Campuran Bahasa & Emoji (Diabaikan)',
        'teks': 'Pak, sy mau izin gak masuk hari ini krn sakit. Sdh ke dokter tdk. Makasih 🙏'
    },
]

print('='*70)
print('🧹 DEMONSTRASI PENANGANAN DIRTY TEXT')
print('='*70)

for i, sample in enumerate(dirty_samples, 1):
    print(f'\n{sample["nama"]}')
    print('-'*60)
    full_preprocessing_pipeline(sample['teks'], verbose=True)
    print()

In [ ]:
# ============================================================
# TERAPKAN PREPROCESSING KE SELURUH DATASET
# ============================================================
df['tokens'] = df['pesan'].apply(full_preprocessing_pipeline)
df['teks_bersih'] = df['tokens'].apply(lambda t: ' '.join(t))
df['jumlah_token'] = df['tokens'].apply(len)

print('📋 HASIL PREPROCESSING DATASET:')
print(df[['id', 'label', 'jumlah_token', 'teks_bersih']].head(10).to_string())
print(f'\n📊 Rata-rata token setelah preprocessing:')
print(df.groupby('label')['jumlah_token'].describe().round(2))

---
## 🔢 4. Perhitungan Manual pada 3 Dokumen Sampel

### Dokumen yang Dipilih:
| | Dokumen | Label |
|--|---------|-------|
| **D1** | "Selamat Anda memenangkan hadiah senilai klaim hadiah sekarang" | SPAM |
| **D2** | "Meeting hari ini jam empat belas ruang rapat laptop" | HAM |
| **D3** | "Promo terbatas iPhone gratis kirim nomor rekening" | SPAM |

### 4.1 Bag of Words (BoW)

In [ ]:
# ============================================================
# PILIH 3 DOKUMEN SAMPEL
# ============================================================
sampel = [
    {"id": "D1", "label": "SPAM",
     "raw": "Selamat! Anda memenangkan hadiah senilai Rp 50.000.000. Klik tautan ini untuk klaim hadiah Anda sekarang!"},
    {"id": "D2", "label": "HAM",
     "raw": "Halo Budi, meeting hari ini jam 14.00 WIB di ruang rapat lantai 3. Jangan lupa bawa laptop."},
    {"id": "D3", "label": "SPAM",
     "raw": "PROMO TERBATAS! Dapatkan iPhone 15 Pro gratis. Kirim nama & nomor rekening ke WA 08123456789"},
]

# Preprocessing dokumen sampel
for doc in sampel:
    doc['tokens'] = full_preprocessing_pipeline(doc['raw'])
    print(f"  [{doc['id']}] ({doc['label']}) → {doc['tokens']}")

# Bangun vocabulary
semua_token = []
for doc in sampel:
    semua_token.extend(doc['tokens'])
vocabulary = sorted(set(semua_token))
vocab_idx = {kata: i for i, kata in enumerate(vocabulary)}

print(f'\n📚 Vocabulary ({len(vocabulary)} term):')
print(f'   {vocabulary}')

In [ ]:
# ============================================================
# 4.1 BAG OF WORDS (BoW) — PERHITUNGAN MANUAL
# ============================================================
print('='*70)
print('📐 METODE 1: BAG OF WORDS (BoW)')
print('='*70)
print()
print('BoW menghitung FREKUENSI kemunculan setiap kata dalam dokumen.')
print('Formula: BoW(t, d) = jumlah kemunculan term t dalam dokumen d')
print()

# Hitung BoW untuk setiap dokumen
bow_matrix = []
for doc in sampel:
    counter = Counter(doc['tokens'])
    bow_vec = [counter.get(kata, 0) for kata in vocabulary]
    bow_matrix.append(bow_vec)
    doc['bow'] = bow_vec

# Tampilkan sebagai DataFrame
df_bow = pd.DataFrame(bow_matrix,
    index=[f"{d['id']} ({d['label']})" for d in sampel],
    columns=vocabulary
)

# Filter hanya kolom yang ada nilainya
df_bow = df_bow.loc[:, (df_bow != 0).any(axis=0)]

print('📊 MATRIKS BOW:')
print(df_bow.to_string())

# Visualisasi heatmap
fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(df_bow, annot=True, fmt='d', cmap='YlOrRd',
    linewidths=0.5, linecolor='gray', ax=ax,
    annot_kws={'size': 9, 'weight': 'bold'})
ax.set_title('Matriks Bag of Words (BoW)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Vocabulary (Terms)', fontsize=11)
ax.set_ylabel('Dokumen', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('../data/bow_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e', edgecolor='none')
plt.show()

### 4.2 TF-IDF (Term Frequency — Inverse Document Frequency)

In [ ]:
# ============================================================
# 4.2 TF-IDF — PERHITUNGAN MANUAL STEP BY STEP
# ============================================================
N = len(sampel)  # Jumlah dokumen

print('='*70)
print('📐 METODE 2: TF-IDF (Term Frequency — Inverse Document Frequency)')
print('='*70)
print()
print('Formula:')
print('  TF(t, d)  = count(t, d) / |d|   [frekuensi relatif term t di dok d]')
print('  IDF(t)    = log(N / df(t))       [log dari total dok / dok berisi t]')
print('  TF-IDF    = TF × IDF')
print()

# --- Hitung TF ---
print('📊 LANGKAH 1: Menghitung TF (Term Frequency)')
print('-'*60)
tf_matrix = []
for doc in sampel:
    counter = Counter(doc['tokens'])
    n_tokens = len(doc['tokens'])
    tf_vec = {}
    for kata in vocabulary:
        tf_val = counter.get(kata, 0) / n_tokens if n_tokens > 0 else 0
        tf_vec[kata] = round(tf_val, 4)
    tf_matrix.append(tf_vec)
    doc['tf'] = tf_vec

df_tf = pd.DataFrame(tf_matrix,
    index=[f"{d['id']} ({d['label']})" for d in sampel]
)
df_tf_display = df_tf.loc[:, (df_tf != 0).any(axis=0)]
print(df_tf_display.to_string())

print()

# --- Hitung DF dan IDF ---
print('📊 LANGKAH 2: Menghitung DF dan IDF')
print('-'*60)
idf = {}
df_count = {}
for kata in vocabulary:
    # Document frequency: berapa dokumen mengandung kata ini
    df_val = sum(1 for doc in sampel if kata in doc['tokens'])
    df_count[kata] = df_val
    # IDF dengan smoothing: log((N+1)/(df+1)) + 1
    idf_val = math.log((N + 1) / (df_val + 1)) + 1
    idf[kata] = round(idf_val, 4)

# Tampilkan DF dan IDF untuk kata-kata yang muncul
df_idf = pd.DataFrame({
    'DF (dok berisi kata)': {k: v for k, v in df_count.items() if v > 0},
    'IDF = log((N+1)/(DF+1))+1': {k: v for k, v in idf.items() if df_count.get(k, 0) > 0}
})
print(f'  Total dokumen (N) = {N}')
print(df_idf.to_string())

print()

# --- Hitung TF-IDF ---
print('📊 LANGKAH 3: Menghitung TF-IDF = TF × IDF')
print('-'*60)
tfidf_matrix = []
for doc in sampel:
    tfidf_vec = {}
    for kata in vocabulary:
        tfidf_val = round(doc['tf'][kata] * idf[kata], 4)
        tfidf_vec[kata] = tfidf_val
    tfidf_matrix.append(tfidf_vec)
    doc['tfidf'] = tfidf_vec

df_tfidf = pd.DataFrame(tfidf_matrix,
    index=[f"{d['id']} ({d['label']})" for d in sampel]
)
df_tfidf_display = df_tfidf.loc[:, (df_tfidf != 0).any(axis=0)]
print(df_tfidf_display.to_string())

# Visualisasi heatmap TF-IDF
fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(df_tfidf_display, annot=True, fmt='.3f', cmap='RdYlGn',
    linewidths=0.5, linecolor='gray', ax=ax,
    annot_kws={'size': 8, 'weight': 'bold'})
ax.set_title('Matriks TF-IDF Score', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Terms (Vocabulary)', fontsize=11)
ax.set_ylabel('Dokumen', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('../data/tfidf_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e', edgecolor='none')
plt.show()

### 4.3 Cosine Similarity

In [ ]:
# ============================================================
# 4.3 COSINE SIMILARITY — PERHITUNGAN MANUAL STEP BY STEP
# ============================================================
print('='*70)
print('📐 METODE 3: COSINE SIMILARITY')
print('='*70)
print()
print('Formula:')
print('  cos(A, B) = (A · B) / (||A|| × ||B||)')
print('  dimana:')
print('    A · B  = Σ(Ai × Bi)           [dot product]')
print('    ||A||  = √(Σ Ai²)             [magnitude/norma vektor A]')
print('    Nilai: 0 = tidak mirip, 1 = identik')
print()

def cosine_similarity_manual(vec_a, vec_b, vocab, nama_a, nama_b):
    """Hitung cosine similarity dengan langkah demi langkah"""
    terms = [k for k in vocab if vec_a.get(k, 0) != 0 or vec_b.get(k, 0) != 0]
    
    print(f'  Pasangan: {nama_a} ↔ {nama_b}')
    print(f'  {"Term":<20} {"TF-IDF A":>10} {"TF-IDF B":>10} {"A×B":>10}')
    print(f'  {"-"*52}')
    
    dot_product = 0
    sum_sq_a = 0
    sum_sq_b = 0
    
    for term in terms:
        a_val = vec_a.get(term, 0)
        b_val = vec_b.get(term, 0)
        axb = a_val * b_val
        dot_product += axb
        sum_sq_a += a_val ** 2
        sum_sq_b += b_val ** 2
        print(f'  {term:<20} {a_val:>10.4f} {b_val:>10.4f} {axb:>10.4f}')
    
    print(f'  {"-"*52}')
    mag_a = math.sqrt(sum_sq_a)
    mag_b = math.sqrt(sum_sq_b)
    
    print(f'  Σ(A×B) [dot product]     = {dot_product:.4f}')
    print(f'  ||A|| = √(Σ A²)          = √{sum_sq_a:.4f} = {mag_a:.4f}')
    print(f'  ||B|| = √(Σ B²)          = √{sum_sq_b:.4f} = {mag_b:.4f}')
    print(f'  ||A|| × ||B||            = {mag_a:.4f} × {mag_b:.4f} = {mag_a*mag_b:.4f}')
    
    if mag_a * mag_b == 0:
        cos_sim = 0.0
    else:
        cos_sim = dot_product / (mag_a * mag_b)
    
    print(f'  ┌────────────────────────────────────────────────┐')
    print(f'  │ cos({nama_a},{nama_b}) = {dot_product:.4f} / {mag_a*mag_b:.4f} = {cos_sim:.4f} │')
    print(f'  └────────────────────────────────────────────────┘')
    
    interpretasi = 'SANGAT MIRIP (keduanya SPAM)' if cos_sim > 0.5 else 'BERBEDA (berbeda label)' if cos_sim < 0.2 else 'AGAK MIRIP'
    print(f'  → Interpretasi: {interpretasi}\n')
    
    return round(cos_sim, 4)

# Hitung similarity untuk 3 pasangan
pasangan = [
    (0, 1, 'D1(SPAM)', 'D2(HAM)'),
    (0, 2, 'D1(SPAM)', 'D3(SPAM)'),
    (1, 2, 'D2(HAM)',  'D3(SPAM)'),
]

similarity_matrix = np.zeros((3, 3))
np.fill_diagonal(similarity_matrix, 1.0)

for i, j, nama_i, nama_j in pasangan:
    print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    sim = cosine_similarity_manual(
        sampel[i]['tfidf'], sampel[j]['tfidf'],
        vocabulary, nama_i, nama_j
    )
    similarity_matrix[i][j] = sim
    similarity_matrix[j][i] = sim

In [ ]:
# Visualisasi Similarity Matrix
fig, ax = plt.subplots(figsize=(7, 5))
labels_doc = [f"{d['id']}\n({d['label']})" for d in sampel]

mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)
sns.heatmap(similarity_matrix, annot=True, fmt='.4f', cmap='coolwarm',
    xticklabels=labels_doc, yticklabels=labels_doc,
    vmin=0, vmax=1, ax=ax, linewidths=2, linecolor='white',
    annot_kws={'size': 14, 'weight': 'bold'})

ax.set_title('Cosine Similarity Matrix\n(Berbasis Vektor TF-IDF)', 
    fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/cosine_similarity_matrix.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e', edgecolor='none')
plt.show()

print('\n📌 KESIMPULAN COSINE SIMILARITY:')
for i, j, nama_i, nama_j in pasangan:
    sim = similarity_matrix[i][j]
    print(f'  {nama_i} ↔ {nama_j}: {sim:.4f}  → {"Mirip" if sim > 0.3 else "Tidak Mirip"}')

---
## 🧪 5. Rencana Eksperimen Milestone 2

### Eksperimen 1: Word-level Feature (BoW) vs Character-level Feature (n-gram)
### Eksperimen 2: Naive Bayes Word-based vs Naive Bayes Character-based

In [ ]:
# ============================================================
# SETUP EKSPERIMEN — PREVIEW KONFIGURASI
# ============================================================

config_eksperimen = {
    'Eksperimen 1': {
        'nama': 'Word-level BoW vs Character N-gram',
        'hipotesis': 'Character n-gram lebih baik menangani typo dan variasi ejaan spam',
        'model': 'Multinomial Naive Bayes',
        'fitur_A': {'metode': 'CountVectorizer', 'analyzer': 'word', 'ngram_range': '(1,1)'},
        'fitur_B': {'metode': 'CountVectorizer', 'analyzer': 'char', 'ngram_range': '(3,5)'},
        'metrik': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
        'split': '75% train / 25% test (stratified)',
    },
    'Eksperimen 2': {
        'nama': 'TF-IDF Word-level vs TF-IDF Character-level',
        'hipotesis': 'TF-IDF memperbaiki kelemahan BoW dengan memberi bobot lebih pada kata diskriminatif',
        'model': 'Multinomial Naive Bayes',
        'fitur_A': {'metode': 'TfidfVectorizer', 'analyzer': 'word', 'ngram_range': '(1,2)'},
        'fitur_B': {'metode': 'TfidfVectorizer', 'analyzer': 'char', 'ngram_range': '(3,5)'},
        'metrik': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Confusion Matrix'],
        'split': '75% train / 25% test (stratified)',
    },
}

print('='*70)
print('🧪 RENCANA 2 EKSPERIMEN MILESTONE 2')
print('='*70)

for key, cfg in config_eksperimen.items():
    print(f'\n  📌 {key}: {cfg["nama"]}')
    print(f'     Hipotesis  : {cfg["hipotesis"]}')
    print(f'     Algoritma  : {cfg["model"]}')
    print(f'     Kondisi A  : {cfg["fitur_A"]}')
    print(f'     Kondisi B  : {cfg["fitur_B"]}')
    print(f'     Metrik     : {cfg["metrik"]}')
    print(f'     Split Data : {cfg["split"]}')

print()
print('='*70)
print('🖥️  RANCANGAN APLIKASI GUI (Streamlit) MILESTONE 2')
print('='*70)
print('''
  Komponen GUI yang Direncanakan:
  ┌─────────────────────────────────────────────────────────────┐
  │  Tab 1: 🔍 Periksa Pesan                                   │
  │    - Input area untuk memasukkan pesan                     │
  │    - Pemilihan model (Eksperimen 1 / Eksperimen 2)         │
  │    - Output: label SPAM/HAM + confidence score            │
  │    - Highlight kata pemicu (trigger words)                 │
  │                                                             │
  │  Tab 2: 📊 Perbandingan Model                              │
  │    - Tabel metrik Eksp 1 vs Eksp 2                        │
  │    - Confusion matrix interaktif                           │
  │    - Visualisasi distribusi confidence                     │
  │                                                             │
  │  Tab 3: 📁 Ekspor & Riwayat                               │
  │    - Download CSV/Excel/JSON riwayat prediksi              │
  │                                                             │
  │  Tab 4: ℹ️  Tentang & Metodologi                           │
  │    - Penjelasan algoritma dan teknik                       │
  └─────────────────────────────────────────────────────────────┘
''')

print('\n✅ Setup eksperimen selesai! Siap untuk Milestone 2.')

---
## 📋 6. Ringkasan & Kesimpulan

| Tahap | Hasil |
|-------|-------|
| **Dataset** | 40 pesan berlabel (spam/ham) bahasa Indonesia & Inggris |
| **Preprocessing** | 6 tahap: lowercase → cleaning → normalisasi → tokenisasi → stopword → stemming |
| **Dirty Text** | Berhasil menangani URL, nomor HP/rekening, capslock, typo, slang |
| **BoW** | Matriks term-frekuensi untuk 3 dokumen sampel |
| **TF-IDF** | Perhitungan TF, IDF, dan TF-IDF step-by-step |
| **Cosine Similarity** | Dokumen spam lebih mirip satu sama lain vs dokumen ham |
| **Next Step** | Eksperimen 2 model di Milestone 2 + Aplikasi Streamlit |
